In [1]:
"""
OceanXRay -- shared geolocation helpers for internal-wave tiles.

Used by infer_iw_on_real_scenes.py (batch run over our scenes) and by
iw_dashboard.py (geolocating a single uploaded tile). Keeping one copy
means the dashboard map and the detections file can never disagree about
where a pixel is.

What it does
------------
1. Parses tile filenames written by convert_scenes_to_images.py:
       {scene_id}_tile{idx}_row{row_start}_col{col_start}.png
   The row/col are the tile's top-left corner in the ORIGINAL scene's
   pixels. Each tile covers 2,500 x 2,500 scene pixels (25 km at 10 m)
   and was resized to 1,024 x 1,024, so 1 tile pixel = 2.44 scene pixels.
2. Converts scene pixels to longitude/latitude with the same GCP-based
   affine transform the rest of the project uses (rasterio's from_gcps,
   a least-squares fit to the scene's ground-control points).
3. Stores that transform as 6 numbers per scene in a small JSON table,
   so the dashboard can geolocate without the multi-GB .SAFE folders.
4. Land / coastline checks against land_ocean_mask.npz.

Build the table (once, wherever the .SAFE folders are, e.g. Kaggle):
    set SAFE_ROOT below, then run this file (works as a script or in a
    notebook cell via %run oceanxray_geo.py). Writes scene_geo_table.json.
    Copy that JSON next to iw_dashboard.py.

Accuracy note: one affine transform per scene is an approximation of the
true SAR geometry (GCP grid). It is the same approximation every map in
this project already uses; treat positions as approximate (sub-km to
~1 km scale), not survey-grade.
"""

import json
import os
import re

import numpy as np

# ---- EDIT THESE (only needed when building the table) ----
SAFE_ROOT = "/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe"                      # folder containing the *.SAFE scene folders
GEO_TABLE_PATH = "scene_geo_table.json"
# ----------------------------------------------------------

TILE_SIZE_KM = 25                     # must match convert_scenes_to_images.py
OUTPUT_PIXELS = 1024                  # must match convert_scenes_to_images.py
NATIVE_PX_M = 10.0                    # Sentinel-1 IW GRD pixel spacing
NATIVE_TILE_PX = int(TILE_SIZE_KM * 1000 / NATIVE_PX_M)   # 2500
TILE_TO_SCENE = NATIVE_TILE_PX / OUTPUT_PIXELS            # 2.44 scene px per tile px
COAST_BUFFER_KM = 1.0                 # same as tier1_fft_backtrack.py

# Region of interest used across the project (lon_min, lat_min, lon_max, lat_max)
ROI_BOUNDS = (69.0, 4.0, 96.0, 24.0)

TILE_FILENAME_RE = re.compile(r"^(.+)_tile(\d+)_row(\d+)_col(\d+)\.png$", re.IGNORECASE)
_SCENE_TIME_RE = re.compile(r"_(\d{8})T(\d{6})_")


# ----------------------------------------------------------------------
# Filenames
# ----------------------------------------------------------------------
def parse_tile_filename(filename):
    """-> (scene_id, tile_idx, row_start, col_start), or None if the name
    doesn't follow the pipeline's pattern (e.g. a benchmark image)."""
    m = TILE_FILENAME_RE.match(os.path.basename(filename))
    if not m:
        return None
    scene_id, tile_idx, row_start, col_start = m.groups()
    return scene_id, int(tile_idx), int(row_start), int(col_start)


def scene_datetime_str(scene_id):
    """'S1A_IW_GRDH_1SDV_20210626T120406_...' -> '2021-06-26 12:04 UTC'."""
    m = _SCENE_TIME_RE.search(scene_id)
    if not m:
        return None
    d, t = m.groups()
    return f"{d[:4]}-{d[4:6]}-{d[6:]} {t[:2]}:{t[2:4]} UTC"


# ----------------------------------------------------------------------
# Transform table
# ----------------------------------------------------------------------
class SceneTransform:
    """Scene pixel (x = column, y = row) -> (lon, lat), from 6 affine numbers
    in rasterio's order: lon = a*x + b*y + c ; lat = d*x + e*y + f."""

    def __init__(self, coeffs, width=None, height=None):
        self.a, self.b, self.c, self.d, self.e, self.f = [float(v) for v in coeffs]
        self.width, self.height = width, height

    def __call__(self, x, y):
        return self.a * x + self.b * y + self.c, self.d * x + self.e * y + self.f

    # Same calling style as rasterio's Affine: T * (x, y)
    def __mul__(self, xy):
        return self(*xy)

    def footprint(self):
        """Scene corners as [[lon, lat], ...] (needs width/height)."""
        if self.width is None or self.height is None:
            return None
        w, h = self.width, self.height
        return [list(self(x, y)) for x, y in ((0, 0), (w, 0), (w, h), (0, h))]


def build_geo_table(safe_root=SAFE_ROOT, out_path=GEO_TABLE_PATH, polarization="vv"):
    """Read every *.SAFE under safe_root, fit the GCP affine, save JSON."""
    import glob
    import rasterio
    from rasterio.transform import from_gcps

    table, failed = {}, []
    safes = sorted(e for e in os.listdir(safe_root) if e.endswith(".SAFE"))
    print(f"Found {len(safes)} .SAFE folders under '{safe_root}'")
    for entry in safes:
        scene_id = entry[:-5]
        hits = glob.glob(os.path.join(safe_root, entry, "measurement", f"*{polarization}*.tiff"))
        if not hits:
            hits = glob.glob(os.path.join(safe_root, entry, "**", "measurement",
                                          f"*{polarization}*.tiff"), recursive=True)
        if not hits:
            failed.append((scene_id, "no measurement tiff"))
            continue
        try:
            with rasterio.open(hits[0]) as src:
                gcps, _ = src.get_gcps()
                if not gcps:
                    raise ValueError("no GCPs")
                A = from_gcps(gcps)
                table[scene_id] = {
                    "affine": [A.a, A.b, A.c, A.d, A.e, A.f],
                    "width": src.width, "height": src.height,
                    "time": scene_datetime_str(scene_id),
                }
        except Exception as e:                      # keep going; report at the end
            failed.append((scene_id, str(e)))
    with open(out_path, "w") as f:
        json.dump(table, f)
    print(f"Saved {len(table)} scene transforms to {out_path}")
    for sid, why in failed:
        print(f"  SKIPPED {sid}: {why}")
    return table


def load_geo_table(path=GEO_TABLE_PATH):
    """-> {scene_id: SceneTransform}, plus the raw dict for metadata."""
    with open(path) as f:
        raw = json.load(f)
    return {sid: SceneTransform(v["affine"], v.get("width"), v.get("height"))
            for sid, v in raw.items()}, raw


# ----------------------------------------------------------------------
# Tile / box geometry
# ----------------------------------------------------------------------
def tile_px_to_scene(row_start, col_start, x, y):
    """Resized-tile pixel (x, y) -> scene pixel (x, y)."""
    return col_start + x * TILE_TO_SCENE, row_start + y * TILE_TO_SCENE


def tile_footprint(T, row_start, col_start):
    """Tile corners as [[lon, lat]] x 4 (TL, TR, BR, BL)."""
    n = OUTPUT_PIXELS
    return [list(T(*tile_px_to_scene(row_start, col_start, x, y)))
            for x, y in ((0, 0), (n, 0), (n, n), (0, n))]


def box_polygon(T, row_start, col_start, box):
    """Detection box [x1, y1, x2, y2] in resized-tile pixels -> 4 corners [[lon, lat]]."""
    x1, y1, x2, y2 = [float(v) for v in box]
    return [list(T(*tile_px_to_scene(row_start, col_start, x, y)))
            for x, y in ((x1, y1), (x2, y1), (x2, y2), (x1, y2))]


def box_center(T, row_start, col_start, box):
    x1, y1, x2, y2 = [float(v) for v in box]
    return T(*tile_px_to_scene(row_start, col_start, (x1 + x2) / 2, (y1 + y2) / 2))


# ----------------------------------------------------------------------
# Land / coast
# ----------------------------------------------------------------------
def load_land_checker(path="land_ocean_mask.npz"):
    """-> is_on_land(lat, lon). Same lookup as the rest of the project."""
    d = np.load(path)
    mask, mlat, mlon = d["mask"], d["lat"], d["lon"]
    dlat, dlon = mlat[1] - mlat[0], mlon[1] - mlon[0]
    lat_lo, lat_hi, lon_lo, lon_hi = mlat.min(), mlat.max(), mlon.min(), mlon.max()

    def is_on_land(lat, lon):
        lat = max(lat_lo, min(lat_hi, lat))
        lon = max(lon_lo, min(lon_hi, lon))
        i = min(int((lat - mlat[0]) / dlat), mask.shape[0] - 1)
        j = min(int((lon - mlon[0]) / dlon), mask.shape[1] - 1)
        return not bool(mask[i, j])          # mask True = ocean
    return is_on_land


def box_touches_land(T, row_start, col_start, box, is_on_land,
                     buffer_km=COAST_BUFFER_KM, n=9):
    """True if any point on an n x n grid over the box, grown by buffer_km,
    is on land. Same rule as tier1_fft_backtrack.py's coastline check."""
    x1, y1, x2, y2 = [float(v) for v in box]
    buf = buffer_km * 1000 / NATIVE_PX_M                       # scene pixels
    sx1, sy1 = tile_px_to_scene(row_start, col_start, x1, y1)
    sx2, sy2 = tile_px_to_scene(row_start, col_start, x2, y2)
    for sy in np.linspace(sy1 - buf, sy2 + buf, n):
        for sx in np.linspace(sx1 - buf, sx2 + buf, n):
            lon, lat = T(sx, sy)
            if is_on_land(lat, lon):
                return True
    return False


# ----------------------------------------------------------------------
# Self-test (no data needed)
# ----------------------------------------------------------------------
def selftest():
    # A made-up scene: north-up, 10 m pixels, top-left at 90E, 20N
    deg_lon, deg_lat = 10 / 104_600, 10 / 110_700
    T = SceneTransform([deg_lon, 0, 90.0, 0, -deg_lat, 20.0], 25000, 16000)
    assert parse_tile_filename("S1A_IW_GRDH_1SDV_20210626T120406_X_tile0073_row12000_col2000.png") \
        == ("S1A_IW_GRDH_1SDV_20210626T120406_X", 73, 12000, 2000)
    assert parse_tile_filename("P0001.png") is None
    assert scene_datetime_str("S1A_IW_GRDH_1SDV_20210626T120406_X") == "2021-06-26 12:04 UTC"
    fp = tile_footprint(T, 0, 0)
    w_km = (fp[1][0] - fp[0][0]) * 104.6
    h_km = (fp[0][1] - fp[3][1]) * 110.7
    assert abs(w_km - 25) < 0.01 and abs(h_km - 25) < 0.01, (w_km, h_km)
    poly = box_polygon(T, 0, 0, [0, 0, 1024, 1024])
    assert np.allclose(poly, fp)
    lon, lat = box_center(T, 0, 0, [0, 0, 1024, 1024])
    assert abs(lon - (90 + 1250 * deg_lon)) < 1e-9 and abs(lat - (20 - 1250 * deg_lat)) < 1e-9
    assert T * (0, 0) == (90.0, 20.0)
    # land: everything west of 90.3E is land in this fake checker
    fake_land = lambda la, lo: lo < 90.3
    assert box_touches_land(T, 0, 0, [0, 0, 100, 100], fake_land)          # box at 90.00E
    assert not box_touches_land(T, 0, 6000, [0, 0, 100, 100], fake_land)   # box at ~90.57E
    # 1 km buffer: box whose west edge sits 0.5 km east of the coast touches land
    col_coast = 0.3 / deg_lon                          # scene px of 90.3E
    col_box = col_coast + 50                           # 500 m east of the coast
    tile_col = int(col_box)                            # put the box at tile x = 0
    assert box_touches_land(T, 0, tile_col, [0, 0, 10, 10], fake_land)
    assert not box_touches_land(T, 0, tile_col, [0, 0, 10, 10], fake_land, buffer_km=0.2)
    print("oceanxray_geo self-test: all checks passed")


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "selftest":
        selftest()
    else:
        build_geo_table()

Found 47 .SAFE folders under '/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe'
Saved 47 scene transforms to scene_geo_table.json


In [2]:
import sys

script_dir = '/kaggle/input/datasets/mayankgupta005/ocean-geo' 

if script_dir not in sys.path:
    sys.path.append(script_dir)

In [3]:
"""
OceanXRay -- Run the IMPROVED internal-wave detector (Faster R-CNN v2)
on our own real Sentinel-1 scene tiles, with every detection geolocated.

What changed from the first version
-----------------------------------
- Model: Faster R-CNN v2 at the full 1024 px input (ckpt_improved_v2_s2.pth,
  the improved-training checkpoint picked on the validation set). The old
  v1 / 800 px checkpoint is no longer used.
- Filters: besides the box-centre land check, each box (grown by 1 km) is
  checked against the land mask on a 9x9 grid -- the coastline rule from
  tier1_fft_backtrack.py. Nothing is silently dropped: every raw detection
  is saved with `on_land` / `touches_coast` flags and a `kept` column, so
  the counts are auditable. Maps use kept == True only.
- Geometry: each detection also stores its box outline as lon/lat corners
  (for drawing on a map), and the run writes scene_geo_table.json (one
  6-number transform per scene) so iw_dashboard.py can geolocate uploads
  without the .SAFE folders.

Expects tiles produced by convert_scenes_to_images.py:
    {scene_id}_tile{idx}_row{row_start}_col{col_start}.png
Needs next to it: oceanxray_geo.py, land_ocean_mask.npz, the checkpoint.

Outputs
-------
real_internal_wave_detections_v2.parquet  -- all raw detections + flags
scene_geo_table.json                       -- per-scene transforms
iw_demo_candidate_filenames.txt            -- tiles with kept detections,
                                              best confidence first
"""

import glob
import os

import pandas as pd
import torch
import torchvision
import torchvision.transforms.functional as TF
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

import oceanxray_geo as geo

# ---- EDIT THESE ----
CHECKPOINT_PATH = "/kaggle/input/models/mayankgupta005/iw-detector-variants/pytorch/default/1/outputs/iw_improve/ckpt_improved_v2_s2.pth"
IMAGE_DIR = "/kaggle/input/datasets/mayankgupta005/sar-image-tiles-june-2021/sar_images_for_iw_detection"     # convert_scenes_to_images.py output
SAFE_ROOT = "/kaggle/input/datasets/andrejkarpathy/sentinel-1-june-2021-raw-safe"                               # folder with the *.SAFE scenes
LAND_MASK_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/land_ocean_mask.npz"
OUTPUT_PATH = "real_internal_wave_detections_v2.parquet"
GEO_TABLE_PATH = "/kaggle/working/scene_geo_table.json"
CONFIDENCE_THRESHOLD = 0.5
NUM_CLASSES = 2
# ---------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model():
    """Same architecture/input size the checkpoint was trained with."""
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None, min_size=1024, max_size=1024)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return model


def load_checkpoint(model, path):
    """improve_iw_detector.py saves a plain state_dict; older scripts saved
    {'model': state_dict, ...}. Accept both."""
    ckpt = torch.load(path, map_location=DEVICE)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(state)
    return model


def main():
    # 1) Per-scene transforms (cheap: reads only GCP metadata)
    if os.path.exists(GEO_TABLE_PATH):
        print(f"Using existing {GEO_TABLE_PATH} (delete it to rebuild)")
    else:
        geo.build_geo_table(SAFE_ROOT, GEO_TABLE_PATH)
    transforms, _ = geo.load_geo_table(GEO_TABLE_PATH)
    is_on_land = geo.load_land_checker(LAND_MASK_PATH)

    # 2) Model
    print("Loading improved detector (Faster R-CNN v2, 1024 px)...")
    model = load_checkpoint(build_model(), CHECKPOINT_PATH).to(DEVICE).eval()

    tile_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
    print(f"Found {len(tile_files)} scene tiles to search")

    rows, n_bad_name, n_no_geo = [], 0, 0
    with torch.no_grad():
        for k, tile_path in enumerate(tile_files):
            filename = os.path.basename(tile_path)
            parsed = geo.parse_tile_filename(filename)
            if parsed is None:
                n_bad_name += 1
                continue
            scene_id, tile_idx, row0, col0 = parsed

            img = Image.open(tile_path).convert("RGB")
            out = model(TF.to_tensor(img).unsqueeze(0).to(DEVICE))[0]
            boxes = out["boxes"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            keep = scores >= CONFIDENCE_THRESHOLD
            if (k + 1) % 200 == 0:
                print(f"  {k + 1}/{len(tile_files)} tiles, {len(rows)} raw detections so far")
            if not keep.any():
                continue

            T = transforms.get(scene_id)
            if T is None:
                n_no_geo += 1

            for box, score in zip(boxes[keep], scores[keep]):
                x1, y1, x2, y2 = [float(v) for v in box]
                sx, sy = geo.tile_px_to_scene(row0, col0, (x1 + x2) / 2, (y1 + y2) / 2)
                rec = {
                    "scene_id": scene_id, "tile_idx": tile_idx, "tile_filename": filename,
                    "confidence": float(score),
                    "box_resized": [x1, y1, x2, y2],
                    "box_x1": x1, "box_y1": y1, "box_x2": x2, "box_y2": y2,
                    "scene_pixel_x": sx, "scene_pixel_y": sy,
                    "real_lon": None, "real_lat": None, "poly_lonlat": None,
                    "on_land": None, "touches_coast": None,
                }
                if T is not None:
                    lon, lat = T(sx, sy)
                    rec["real_lon"], rec["real_lat"] = float(lon), float(lat)
                    rec["poly_lonlat"] = [float(v) for pt in
                                          geo.box_polygon(T, row0, col0, box) for v in pt]
                    rec["on_land"] = bool(is_on_land(lat, lon))
                    rec["touches_coast"] = bool(rec["on_land"] or geo.box_touches_land(
                        T, row0, col0, box, is_on_land))
                rows.append(rec)

    if not rows:
        print("No detections above threshold in any tile.")
        return

    df = pd.DataFrame(rows)
    df["kept"] = df["touches_coast"].eq(False)     # geolocated AND clear of land by 1 km
    df.to_parquet(OUTPUT_PATH, index=False)

    n = len(df)
    n_land = int(df["on_land"].eq(True).sum())
    n_coast = int((df["touches_coast"].eq(True) & ~df["on_land"].eq(True)).sum())
    kept = df[df["kept"]]
    print("\n" + "=" * 60)
    print(f"Raw detections (score >= {CONFIDENCE_THRESHOLD}): {n}")
    print(f"  centre on land:                {n_land} ({100 * n_land / n:.1f}%)")
    print(f"  box within 1 km of coast:      {n_coast} ({100 * n_coast / n:.1f}%)")
    print(f"  not geolocated (no transform): {int(df['real_lat'].isna().sum())}")
    print(f"KEPT (open water, >1 km from land): {len(kept)} in {kept['scene_id'].nunique()} scenes")
    print("=" * 60)
    if n_bad_name:
        print(f"Skipped {n_bad_name} files not matching the tile naming pattern")
    if n_no_geo:
        print(f"{n_no_geo} tiles had detections but no scene transform "
              f"(scene missing from {GEO_TABLE_PATH})")
    if len(kept):
        print(kept["confidence"].describe().round(3).to_string())
        print(f"lat {kept['real_lat'].min():.2f} to {kept['real_lat'].max():.2f}, "
              f"lon {kept['real_lon'].min():.2f} to {kept['real_lon'].max():.2f}")
        best = (kept.groupby("tile_filename")["confidence"].max()
                .sort_values(ascending=False))
        with open("iw_demo_candidate_filenames.txt", "w") as f:
            f.write("\n".join(f"{name}\t{conf:.3f}" for name, conf in best.items()))
        print(f"{len(best)} tiles with kept detections -> iw_demo_candidate_filenames.txt "
              f"(best first; good dashboard demo inputs)")
    print(f"Saved {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Using existing /kaggle/working/scene_geo_table.json (delete it to rebuild)
Loading improved detector (Faster R-CNN v2, 1024 px)...
Found 4524 scene tiles to search
  200/4524 tiles, 139 raw detections so far
  400/4524 tiles, 275 raw detections so far
  600/4524 tiles, 439 raw detections so far
  800/4524 tiles, 599 raw detections so far
  1000/4524 tiles, 702 raw detections so far
  1200/4524 tiles, 789 raw detections so far
  1400/4524 tiles, 916 raw detections so far
  1600/4524 tiles, 1206 raw detections so far
  1800/4524 tiles, 1471 raw detections so far
  2000/4524 tiles, 1671 raw detections so far
  2200/4524 tiles, 1834 raw detections so far
  2400/4524 tiles, 2041 raw detections so far
  2600/4524 tiles, 2280 raw detections so far
  2800/4524 tiles, 2451 raw detections so far
  3000/4524 tiles, 2765 raw detections so far
  3200/4524 tiles, 3022 raw detections so far
  3400/4524 tiles, 3215 raw detections so far
  3600/4524 tiles, 3427 raw detections so far
  3800/4524 tiles, 

In [4]:
%pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
%%writefile /kaggle/working/oceanxray_context.py
"""
OceanXRay -- place names and scene-time weather for the dashboard.

Two small helpers, one file:

1. PLACES (works offline). Turns coordinates into words:
       Andaman Sea · 38 km E of Little Andaman (India)
   The sea name comes from Natural Earth marine polygons; the reference
   point is the nearest named island coast or town. Open ocean has no
   street address, so "sea + nearest land" is the honest description.
   Data (public domain, Natural Earth: seas, towns, island coasts) lives in
   one small file next to this script: oceanxray_context_data.bin
   (zlib-compressed JSON, about 25 KB). It is looked for in the same folder
   as this file, the current folder, /kaggle/working and under /kaggle/input;
   set OCEANXRAY_CONTEXT_DATA to a full path to override. If it is missing,
   importing this module raises ImportError, so the dashboard falls back to
   plain coordinates.

2. CONDITIONS (needs internet). Wind, waves and sea-surface temperature at
   the scene's date and hour, from the Open-Meteo historical API (ERA5
   reanalysis -- the same family of data the project already uses). These
   are modelled values on a coarse grid (about 25 km), not measurements at
   the exact spot. fetch_conditions() never raises: it returns
   (values or None, error text or None) so the dashboard can fall back.

Run `python oceanxray_context.py selftest` to check the place lookup.
"""

import glob
import json
import math
import os
import zlib
from datetime import datetime, timedelta, timezone

import numpy as np

# ----------------------------------------------------------------------
# Places
# ----------------------------------------------------------------------
_D = None
_FILE = "oceanxray_context_data.bin"


def _find_data_file():
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    cands = [os.environ.get("OCEANXRAY_CONTEXT_DATA", ""), os.path.join(here, _FILE),
             os.path.join(os.getcwd(), _FILE), os.path.join("/kaggle/working", _FILE)]
    cands += glob.glob(os.path.join("/kaggle/input", "**", _FILE), recursive=True)
    for c in cands:
        if c and os.path.isfile(c):
            return c
    return None


_DATA_PATH = _find_data_file()
if _DATA_PATH is None:
    raise ImportError(f"{_FILE} not found: put it next to oceanxray_context.py "
                      "(or set OCEANXRAY_CONTEXT_DATA)")


def _data():
    global _D
    if _D is None:
        with open(_DATA_PATH, "rb") as f:
            raw = json.loads(zlib.decompress(f.read()))
        seas = []
        for s in raw["seas"]:
            rings = [np.asarray(r, dtype=float) for r in s["r"]]
            area = sum(abs(_shoelace(r)) for r in rings)
            seas.append((s["n"], rings, area))
        pts = raw["pts"]
        _D = {
            "seas": seas,
            "lon": np.array([p[0] for p in pts]), "lat": np.array([p[1] for p in pts]),
            "name": [p[2] for p in pts], "cc": [p[3] for p in pts], "kind": [p[4] for p in pts],
        }
    return _D


def _shoelace(r):
    x, y = r[:, 0], r[:, 1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def _in_ring(lon, lat, r):
    x, y = r[:, 0], r[:, 1]
    x2, y2 = np.roll(x, -1), np.roll(y, -1)
    cross = ((y > lat) != (y2 > lat)) & (lon < (x2 - x) * (lat - y) / (y2 - y + 1e-12) + x)
    return bool(np.count_nonzero(cross) % 2)


def sea_name(lat, lon):
    """Smallest named sea / bay / gulf containing the point ('Indian Ocean' if none smaller)."""
    best, best_area = None, float("inf")
    for name, rings, area in _data()["seas"]:
        if area < best_area and any(
                r[:, 0].min() <= lon <= r[:, 0].max() and r[:, 1].min() <= lat <= r[:, 1].max()
                and _in_ring(lon, lat, r) for r in rings):
            best, best_area = name, area
    return best or "Indian Ocean"


def _km(lat0, lon0, lat1, lon1):
    p = math.pi / 180
    a = (np.sin((lat1 - lat0) * p / 2) ** 2
         + np.cos(lat0 * p) * np.cos(lat1 * p) * np.sin((lon1 - lon0) * p / 2) ** 2)
    return 12742.0 * np.arcsin(np.sqrt(a))


_DIRS8 = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
_DIRS16 = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
           "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]


def compass(deg, points=8):
    names = _DIRS8 if points == 8 else _DIRS16
    step = 360 / len(names)
    return names[int(((deg % 360) + step / 2) // step) % len(names)]


def _bearing(lat0, lon0, lat1, lon1):
    """Compass bearing FROM point 0 TO point 1, degrees."""
    p = math.pi / 180
    dl = (lon1 - lon0) * p
    y = math.sin(dl) * math.cos(lat1 * p)
    x = math.cos(lat0 * p) * math.sin(lat1 * p) - math.sin(lat0 * p) * math.cos(lat1 * p) * math.cos(dl)
    return (math.atan2(y, x) / p) % 360


def nearest_landmarks(lat, lon, n=3):
    """Nearest distinct named places, nearest first. Each: name, country, km,
    dir (where the query point lies relative to the landmark), lat/lon of the landmark point."""
    d = _data()
    dist = _km(lat, lon, d["lat"], d["lon"])
    out, seen = [], set()
    for i in np.argsort(dist):
        nm = d["name"][i]
        if nm in seen:
            continue
        seen.add(nm)
        out.append({"name": nm, "country": d["cc"][i], "kind": d["kind"][i], "km": float(dist[i]),
                    "dir": compass(_bearing(float(d["lat"][i]), float(d["lon"][i]), lat, lon)),
                    "lat": float(d["lat"][i]), "lon": float(d["lon"][i])})
        if len(out) >= n:
            break
    return out


def describe_point(lat, lon):
    """-> {'sea', 'near' (dict), 'text'}  e.g. 'Andaman Sea · 38 km E of Little Andaman (India)'."""
    sea = sea_name(lat, lon)
    near = nearest_landmarks(lat, lon, 1)[0]
    km = near["km"]
    dist_txt = "at" if km < 1 else f"{km:.0f} km {near['dir']} of"
    country = f" ({near['country']})" if near["country"] and near["country"] != near["name"] else ""
    text = f"{sea} · {dist_txt} {near['name']}{country}".replace("at  ", "at ")
    return {"sea": sea, "near": near, "text": text}


_ANDAMAN_NICOBAR = {"Great Nicobar", "South Andaman", "Middle Andaman", "North Andaman", "Car Nicobar",
                    "Nancowry", "Katchal", "Little Nicobar", "Campbell Bay", "Little Andaman",
                    "Havelock Island", "Neil Island", "Barren Island", "Narcondam Island",
                    "Landfall Island", "Port Blair", "Diglipur", "Mayabunder"}


# ----------------------------------------------------------------------
# Conditions at scene time
# ----------------------------------------------------------------------
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MARINE_URL = "https://marine-api.open-meteo.com/v1/marine"


def parse_scene_time(text):
    """'2021-06-26 12:04 UTC' -> aware datetime, or None."""
    try:
        return datetime.strptime(text.replace(" UTC", ""), "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
    except (AttributeError, ValueError):
        return None


def _nearest_hour(when):
    h = when.replace(minute=0, second=0, microsecond=0)
    return h + timedelta(hours=1) if when.minute >= 30 else h


def _pick(block, key, idx):
    try:
        v = block[key][idx]
        return None if v is None else float(v)
    except (KeyError, IndexError, TypeError, ValueError):
        return None


def fetch_conditions(lat, lon, when_utc, timeout=8, http_get=None):
    """-> (dict of values or None, error text or None). Values may be None individually
    (e.g. no wave data over land). `http_get(url, params, timeout)` -> parsed JSON is
    injectable for tests."""
    if http_get is None:
        import requests

        def http_get(url, params, timeout):
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()

    hour = _nearest_hour(when_utc)
    key = hour.strftime("%Y-%m-%dT%H:00")
    day0 = hour.strftime("%Y-%m-%d")
    day1 = (hour + timedelta(days=1)).strftime("%Y-%m-%d")
    base = {"latitude": round(lat, 3), "longitude": round(lon, 3), "start_date": day0,
            "end_date": day1, "timezone": "GMT"}
    out, errors = {"hour_utc": key}, []

    def hourly(url, extra):
        j = http_get(url, {**base, **extra}, timeout)
        h = j["hourly"]
        return h, h["time"].index(key)

    try:
        h, i = hourly(ARCHIVE_URL, {
            "hourly": "wind_speed_10m,wind_direction_10m,wind_gusts_10m,pressure_msl,cloud_cover,precipitation",
            "wind_speed_unit": "ms"})
        out.update(wind_ms=_pick(h, "wind_speed_10m", i), wind_dir=_pick(h, "wind_direction_10m", i),
                   gust_ms=_pick(h, "wind_gusts_10m", i), pressure_hpa=_pick(h, "pressure_msl", i),
                   cloud_pct=_pick(h, "cloud_cover", i), rain_mm=_pick(h, "precipitation", i))
    except Exception as e:                      # network, HTTP error, unexpected shape
        errors.append(f"wind/weather: {type(e).__name__}: {e}")
        if isinstance(e, OSError) and getattr(e, "response", None) is None:
            return None, errors[0]              # no connection at all: don't wait on more requests

    marine = None
    for variables in ("wave_height,wave_period,wave_direction,sea_surface_temperature",
                      "wave_height,wave_period,wave_direction"):      # retry without SST if rejected
        try:
            marine = hourly(MARINE_URL, {"hourly": variables})
            break
        except Exception as e:
            err = f"waves/sea temperature: {type(e).__name__}: {e}"
    if marine is None:
        errors.append(err)
    else:
        h, i = marine
        out.update(wave_h=_pick(h, "wave_height", i), wave_t=_pick(h, "wave_period", i),
                   wave_dir=_pick(h, "wave_direction", i), sst_c=_pick(h, "sea_surface_temperature", i))

    has_any = any(out.get(k) is not None for k in
                  ("wind_ms", "wave_h", "sst_c", "pressure_hpa", "cloud_pct"))
    if not has_any:
        return None, "; ".join(errors) or "no data returned"
    out["source"] = "Open-Meteo historical (ERA5 reanalysis)"
    return out, ("; ".join(errors) or None)


def interpret(cond, place=None):
    """Short plain-language reading of the conditions. Rules of thumb, hedged."""
    lines = []
    w = cond.get("wind_ms") if cond else None
    if w is not None:
        if w < 2:
            lines.append(f"Wind was very light ({w:.1f} m/s). A glassy sea gives radar little contrast, "
                         "so internal waves can be hard to see and some may be missed.")
        elif w <= 10:
            lines.append(f"Wind was moderate ({w:.1f} m/s), the range in which internal waves "
                         "usually show up well in radar images.")
        else:
            lines.append(f"Wind was strong ({w:.1f} m/s). Rough water can hide internal-wave "
                         "patterns, so weaker waves may be missed.")
    if place:
        near = place["near"]
        if near["name"] in _ANDAMAN_NICOBAR and near["km"] <= 250:
            lines.append(f"This is {near['km']:.0f} km {near['dir']} of {near['name']}, on the "
                         "Andaman-Nicobar island chain. Tides flowing over this underwater ridge are a "
                         "known source of strong internal waves in the Andaman Sea.")
        else:
            lines.append("Internal waves in this region are usually generated where tides flow over "
                         "the continental shelf edge or underwater ridges.")
    return lines


# ----------------------------------------------------------------------
def selftest():
    p = describe_point(8.0, 94.6)                      # east of Nicobar Is. (Andaman Sea side)
    assert p["sea"] in ("Andaman Sea", "Bay of Bengal"), p
    assert p["near"]["name"] in _ANDAMAN_NICOBAR | {"Nancowry", "Katchal"}, p
    p2 = describe_point(15.0, 87.0)
    assert p2["sea"] == "Bay of Bengal", p2
    assert describe_point(10.5, 66.0)["sea"] in ("Arabian Sea", "Indian Ocean")
    assert compass(0) == "N" and compass(225) == "SW" and compass(359) == "N"
    t = parse_scene_time("2021-06-26 12:04 UTC")
    assert _nearest_hour(t).hour == 12 and _nearest_hour(t.replace(minute=40)).hour == 13
    assert parse_scene_time(None) is None

    def fake_get(url, params, timeout):
        times = [f"2021-06-26T{h:02d}:00" for h in range(24)] + [f"2021-06-27T{h:02d}:00" for h in range(24)]
        n = len(times)
        if "archive" in url:
            return {"hourly": {"time": times, "wind_speed_10m": [4.0] * n, "wind_direction_10m": [225.0] * n,
                               "wind_gusts_10m": [6.0] * n, "pressure_msl": [1008.0] * n,
                               "cloud_cover": [60] * n, "precipitation": [0.2] * n}}
        if "sea_surface_temperature" in params["hourly"]:
            raise RuntimeError("400 unknown variable")      # exercises the retry
        return {"hourly": {"time": times, "wave_height": [1.4] * n, "wave_period": [7.0] * n,
                           "wave_direction": [200.0] * n}}
    c, err = fetch_conditions(8.0, 94.6, t, http_get=fake_get)
    assert c["wind_ms"] == 4.0 and c["wave_h"] == 1.4 and c["sst_c"] is None, c
    assert c["hour_utc"] == "2021-06-26T12:00"
    c2, err2 = fetch_conditions(8.0, 94.6, t, http_get=lambda *a: (_ for _ in ()).throw(OSError("offline")))
    assert c2 is None and "offline" in err2
    assert len(interpret(c, p)) == 2
    print("oceanxray_context self-test: all checks passed")
    print(p["text"]); print(p2["text"])




if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "selftest":
        selftest()
    else:
        print(describe_point(float(sys.argv[1]), float(sys.argv[2]))["text"] if len(sys.argv) == 3
              else "usage: python oceanxray_context.py selftest | <lat> <lon>")

Writing /kaggle/working/oceanxray_context.py


In [6]:
%%writefile /kaggle/working/oceanxray_bathy.py
"""oceanxray_bathy.py -- sea-floor depth for any point in the study area (GEBCO).

Needs one data file next to this script: oceanxray_bathy_grid.bin (0.05 degree, about 5.5 km,
GEBCO grid for 4-24 N, 68-94 E; zlib-compressed int16, 5 m steps; about 220 KB).
It is looked for in the same folder as this file, the current folder, /kaggle/working, and under
/kaggle/input; set the environment variable OCEANXRAY_BATHY_GRID to a full path to override.
GEBCO Compilation Group (2026) GEBCO Grid. https://www.gebco.net  (terms: free to use with attribution)

    depth_at(lat, lon)        -> metres below sea level (negative = land) or None outside the grid
    describe_seafloor(lat, lon) -> dict(depth_m, zone, slope_m_per_km, shelf_break_km, text)
"""
import glob
import math
import os
import zlib

import numpy as np

_N, _M, _LA0, _LO0, _STEP = 400, 520, 4.025, 68.025, 0.05
_GRID = None
_FILE = "oceanxray_bathy_grid.bin"


def _find_grid_file():
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    cands = [os.environ.get("OCEANXRAY_BATHY_GRID", ""), os.path.join(here, _FILE),
             os.path.join(os.getcwd(), _FILE), os.path.join("/kaggle/working", _FILE)]
    cands += glob.glob(os.path.join("/kaggle/input", "**", _FILE), recursive=True)
    for c in cands:
        if c and os.path.isfile(c):
            return c
    return None


def _grid():
    """Loads the grid once; returns None (and the depth panel simply stays hidden) if the file is missing."""
    global _GRID
    if _GRID is None:
        path = _find_grid_file()
        if path is None:
            return None
        with open(path, "rb") as f:
            q = np.frombuffer(zlib.decompress(f.read()), dtype=np.int16).reshape(_N, _M)
        _GRID = q.astype(np.float32) * 5.0          # metres, negative = below sea level
    return _GRID


def _ij(lat, lon):
    return (lat - _LA0) / _STEP, (lon - _LO0) / _STEP


def depth_at(lat, lon):
    g = _grid()
    if g is None:
        return None
    fi, fj = _ij(lat, lon)
    if not (0 <= fi < _N - 1 and 0 <= fj < _M - 1):
        return None
    i, j = int(fi), int(fj)
    a, b = fi - i, fj - j
    z = (g[i, j] * (1 - a) * (1 - b) + g[i + 1, j] * a * (1 - b)
         + g[i, j + 1] * (1 - a) * b + g[i + 1, j + 1] * a * b)
    return float(-z)


def _zone(d):
    if d <= 0:
        return "land"
    if d < 50:
        return "very shallow shelf"
    if d < 200:
        return "continental shelf"
    if d < 1000:
        return "upper slope"
    if d < 3000:
        return "deep slope"
    return "deep ocean"


def describe_seafloor(lat, lon, radius_km=15.0, search_km=150.0):
    d = depth_at(lat, lon)
    if d is None:
        return None                    # outside the grid, or the .bin file was not found
    g = _grid()
    fi, fj = _ij(lat, lon)
    ci, cj = int(round(fi)), int(round(fj))
    kmpx = _STEP * 111.0
    r = max(1, int(radius_km / kmpx))
    sl = g[max(ci - r, 1):ci + r + 1, max(cj - r, 1):cj + r + 1]
    gy, gx = np.gradient(sl, kmpx)
    slope = float(np.nanmax(np.hypot(gx, gy))) if sl.size else 0.0
    # distance to the 200 m contour (shelf break)
    R = int(search_km / kmpx)
    i0, i1 = max(ci - R, 0), min(ci + R + 1, _N)
    j0, j1 = max(cj - R, 0), min(cj + R + 1, _M)
    win = -g[i0:i1, j0:j1]
    near = np.abs(win - 200.0) < 40.0
    sb = None
    if near.any():
        ii, jj = np.nonzero(near)
        dy = (ii + i0 - fi) * kmpx
        dx = (jj + j0 - fj) * kmpx * math.cos(math.radians(lat))
        sb = float(np.min(np.hypot(dx, dy)))
    zone = _zone(d)
    if d <= 0:
        text = "on land in the depth grid"
    else:
        text = f"sea floor about {d:,.0f} m deep ({zone})"
        if sb is not None and sb <= 60 and d >= 20:
            text += f", about {sb:.0f} km from the shelf edge (200 m depth)"
        if slope >= 40:
            text += f"; steep sea-floor slope nearby (up to {slope:.0f} m per km)"
    return {"depth_m": d, "zone": zone, "slope_m_per_km": slope, "shelf_break_km": sb, "text": text}


def selftest():
    assert _grid() is not None, f"{_FILE} not found next to this script"
    assert abs(depth_at(21.5, 90.8) - 20) < 40                      # shallow northern Bay of Bengal shelf
    assert depth_at(19.7, 89.2) > 800                               # Swatch of No Ground, canyon head area
    assert depth_at(23.0, 80.0) is None or depth_at(23.0, 80.0) < 0  # land (India)
    assert depth_at(0.0, 80.0) is None                              # outside grid
    s = describe_seafloor(20.6, 89.1)
    assert s and s["depth_m"] > 100, s
    print("oceanxray_bathy self-test: all checks passed")
    for p in ((21.5, 90.8), (20.6, 89.1), (8.0, 93.5), (13.0, 85.0)):
        print(p, describe_seafloor(*p)["text"])



if __name__ == "__main__":
    selftest()

Writing /kaggle/working/oceanxray_bathy.py


In [7]:
%%writefile /kaggle/working/iw_dashboard.py
"""
OceanXRay -- Internal Wave Detection Dashboard

Upload a Sentinel-1 tile, run the improved detector (Faster R-CNN v2,
92.3% mAP@0.5 on the S1-IW-2023 test split), see captioned boxes, and --
for tiles from our own pipeline -- watch the map fly from the study region
to where the tile actually is.

Run with: streamlit run iw_dashboard.py

Files expected next to this script
----------------------------------
required  ckpt_improved_v2_s2.pth                     the detector
optional  oceanxray_geo.py + scene_geo_table.json     geolocating uploads
optional  land_ocean_mask.npz                         'near land' warnings
optional  oceanxray_context.py                        place names + weather at scene time
optional  real_internal_wave_detections_v2.parquet    detection spots on the maps
          (falls back to real_internal_wave_detections.parquet, old model)
optional  sar_era5_sst_spatial_matches_june.parquet   SST/wind map tab

Speed notes
-----------
Streamlit re-runs this whole script on every click, including the code of
tabs you are not looking at. So: the model is loaded once (cache_resource),
detector output is cached per uploaded file (moving the threshold slider
only re-filters boxes, it never re-runs the model), and the cartopy map is
drawn once and reused as a PNG.
"""

import hashlib
import io
from html import escape as html_escape
import json
import os

import numpy as np
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
import torch
import torchvision
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

try:
    import oceanxray_geo as geo
except ImportError:            # dashboard still works, just without geolocation
    geo = None

try:
    import oceanxray_context as ctx   # place names + scene-time weather
except ImportError:                   # dashboard still works, with coordinates only
    ctx = None

try:
    import oceanxray_bathy as bathy   # GEBCO sea-floor depth (embedded grid)
except ImportError:                   # dashboard still works without the depth panel
    bathy = None

st.set_page_config(page_title="OceanXRay -- Internal Wave Detector", layout="wide")

CHECKPOINT_PATH = "/kaggle/input/models/mayankgupta005/iw-detector-variants/pytorch/default/1/outputs/iw_improve/ckpt_improved_v2_s2.pth"
GEO_TABLE_PATH = "/kaggle/input/datasets/mayankgupta005/improve-landing-inputs/scene_geo_table.json"
LAND_MASK_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/land_ocean_mask.npz"
SPATIAL_MATCHES_PATH = "/kaggle/input/datasets/mayankgupta005/sar-metadata/sar_era5_sst_spatial_matches_june.parquet"
DETECTIONS_V2_PATH = "/kaggle/input/datasets/mayankgupta005/improve-landing-inputs/real_internal_wave_detections_v2.parquet"
DETECTIONS_OLD_PATH = "/kaggle/input/datasets/mayankgupta005/internal-wave-detection-metadata/real_internal_wave_detections.parquet"
BOX_COLOR = (255, 196, 0)          # amber: readable on grey SAR
NEAR_LAND_COLOR = (156, 163, 175)  # grey: box within 1 km of land
NUM_CLASSES = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROI = (69.0, 4.0, 96.0, 24.0)      # lon_min, lat_min, lon_max, lat_max


# ----------------------------------------------------------------------
# Cached resources (built once per server process)
# ----------------------------------------------------------------------
@st.cache_resource(show_spinner="Loading the detector...")
def load_model():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None, min_size=1024, max_size=1024)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


@st.cache_resource
def load_geo():
    """-> (transforms, raw table, is_on_land or None), or None if unavailable."""
    if geo is None or not os.path.exists(GEO_TABLE_PATH):
        return None
    transforms, raw = geo.load_geo_table(GEO_TABLE_PATH)
    land = geo.load_land_checker(LAND_MASK_PATH) if os.path.exists(LAND_MASK_PATH) else None
    return transforms, raw, land


@st.cache_data(show_spinner="Running the detector...", max_entries=32)
def detect(image_bytes: bytes):
    """All boxes + scores for one image. Cached by file content, so changing
    the threshold (or any other widget) never re-runs the model."""
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    with torch.no_grad():
        out = load_model()(TF.to_tensor(image).unsqueeze(0).to(DEVICE))[0]
    return out["boxes"].cpu().numpy(), out["scores"].cpu().numpy(), image.size


def files_stamp(*paths):
    """Modification times, passed to cached functions so they refresh when a file changes."""
    return tuple(os.path.getmtime(p) if os.path.exists(p) else None for p in paths)


@st.cache_data
def load_detections(stamp=None):
    """Detection spots for the maps -> (DataFrame[lat, lon, confidence, scene_id], note)."""
    if os.path.exists(DETECTIONS_V2_PATH):
        df = pd.read_parquet(DETECTIONS_V2_PATH)
        df = df[df["kept"]] if "kept" in df else df
        note = "improved model, open water only (>1 km from land)"
    elif os.path.exists(DETECTIONS_OLD_PATH):
        df = pd.read_parquet(DETECTIONS_OLD_PATH)
        note = ("OLD model output, includes coastal false positives -- "
                "re-run infer_iw_on_real_scenes.py to refresh")
    else:
        return pd.DataFrame(columns=["real_lat", "real_lon", "confidence", "scene_id"]), "none found"
    df = df.dropna(subset=["real_lat", "real_lon"])
    return df[["real_lat", "real_lon", "confidence", "scene_id"]].reset_index(drop=True), note


# ----------------------------------------------------------------------
# Drawing
# ----------------------------------------------------------------------
def _font(size):
    for f in ("DejaVuSans-Bold.ttf", "arialbd.ttf", "Arial Bold.ttf"):
        try:
            return ImageFont.truetype(f, size)
        except OSError:
            pass
    return ImageFont.load_default()


def trim_nodata(img, dets):
    """Crop away all-black rows/columns (outside the radar swath) for display only.

    Returns the cropped image and the detections with boxes shifted to match. The
    detection itself always runs on the full, unmodified tile."""
    a = np.asarray(img.convert("L"))
    valid = a > 3
    rows, cols = np.where(valid.any(axis=1))[0], np.where(valid.any(axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        return img, dets
    y0, y1, x0, x1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    for d in dets:                                          # never crop away a detection box
        bx1, by1, bx2, by2 = d["box"]
        x0, y0 = min(x0, int(bx1)), min(y0, int(by1))
        x1, y1 = max(x1, int(np.ceil(bx2))), max(y1, int(np.ceil(by2)))
    x0, y0 = max(x0, 0), max(y0, 0)
    x1, y1 = min(x1, a.shape[1]), min(y1, a.shape[0])
    if (y1 - y0) * (x1 - x0) > 0.95 * a.shape[0] * a.shape[1]:
        return img, dets                                   # nothing worth trimming
    shifted = []
    for d in dets:
        bx1, by1, bx2, by2 = d["box"]
        shifted.append(dict(d, box=[bx1 - x0, by1 - y0, bx2 - x0, by2 - y0]))
    return img.crop((x0, y0, x1, y1)), shifted


def draw_detections(img, dets):
    """Box + filled caption tag 'Internal wave 87%' (grey + 'near land' if flagged)."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    W = img.width
    font = _font(max(14, W // 45))
    lw = max(3, W // 300)
    for d in dets:
        x1, y1, x2, y2 = d["box"]
        color = NEAR_LAND_COLOR if d["near_land"] else BOX_COLOR
        draw.rectangle([x1, y1, x2, y2], outline=color, width=lw)
        label = f"Internal wave {d['score'] * 100:.0f}%" + (" · near land" if d["near_land"] else "")
        tx0, ty0, tx1, ty1 = draw.textbbox((0, 0), label, font=font)
        tw, th = tx1 - tx0, ty1 - ty0
        pad = max(3, th // 4)
        ty = y1 - th - 2 * pad if y1 - th - 2 * pad >= 0 else y1
        tx = min(x1, W - tw - 2 * pad)
        draw.rectangle([tx, ty, tx + tw + 2 * pad, ty + th + 2 * pad], fill=color)
        draw.text((tx + pad, ty + pad - ty0), label, fill="white", font=font)
    return img


def place_for(lon, lat):
    """{'sea', 'near', 'text'} for a point, or None if the context module is missing."""
    if ctx is None:
        return None
    try:
        return ctx.describe_point(lat, lon)
    except Exception:
        return None


def view_box(tile_fp, landmarks, min_deg=0.9):
    """Map view that holds the tile and its nearest landmark(s): [[south, west], [north, east]].
    At least ~100 km across so the tile is never filling the screen."""
    lons = [p[0] for p in tile_fp] + [m["lon"] for m in landmarks]
    lats = [p[1] for p in tile_fp] + [m["lat"] for m in landmarks]
    w, e, s, n = min(lons), max(lons), min(lats), max(lats)
    if e - w < min_deg:
        c = (e + w) / 2
        w, e = c - min_deg / 2, c + min_deg / 2
    if n - s < min_deg:
        c = (n + s) / 2
        s, n = c - min_deg / 2, c + min_deg / 2
    return [[s, w], [n, e]]


def geolocate(filename, image_size, boxes, scores, threshold):
    """-> (dets, tile_info or None, reason-if-not-geolocated)."""
    keep = scores >= threshold
    dets = [{"box": [float(v) for v in b], "score": float(s), "near_land": False, "poly": None}
            for b, s in zip(boxes[keep], scores[keep])]
    G = load_geo()
    parsed = geo.parse_tile_filename(filename) if geo is not None else None
    if parsed is None:
        return dets, None, ("This file name isn't one of our pipeline tiles "
                            "(`<scene>_tileNNNN_rowR_colC.png`), so its location is unknown.")
    if G is None:
        return dets, None, (f"Geolocation needs `oceanxray_geo.py` and `{GEO_TABLE_PATH}` "
                            "next to the dashboard.")
    transforms, raw, is_on_land = G
    scene_id, tile_idx, row0, col0 = parsed
    T = transforms.get(scene_id)
    if T is None:
        return dets, None, f"Scene `{scene_id[:40]}…` is not in `{GEO_TABLE_PATH}`."

    # Boxes come back in the uploaded image's pixels; the geometry assumes
    # the pipeline's 1024 px tiles, so rescale if the upload was resized.
    sx = geo.OUTPUT_PIXELS / image_size[0]
    sy = geo.OUTPUT_PIXELS / image_size[1]
    for d in dets:
        x1, y1, x2, y2 = d["box"]
        b = [x1 * sx, y1 * sy, x2 * sx, y2 * sy]
        d["poly"] = geo.box_polygon(T, row0, col0, b)
        d["center"] = geo.box_center(T, row0, col0, b)
        if is_on_land is not None:
            d["near_land"] = bool(geo.box_touches_land(T, row0, col0, b, is_on_land))
        d["place"] = place_for(*d["center"])
    center = geo.box_center(T, row0, col0, [0, 0, geo.OUTPUT_PIXELS, geo.OUTPUT_PIXELS])
    tile_fp = geo.tile_footprint(T, row0, col0)
    landmarks = []
    if ctx is not None:
        try:
            landmarks = [m for m in ctx.nearest_landmarks(center[1], center[0], 3) if m["km"] <= 220]
        except Exception:
            landmarks = []
    info = {
        "scene_fp": T.footprint(),
        "tile_fp": tile_fp,
        "center": center,
        "time": raw[scene_id].get("time") or geo.scene_datetime_str(scene_id),
        "tile_idx": tile_idx,
        "place": place_for(*center),
        "landmarks": landmarks,
        "view": view_box(tile_fp, landmarks[:2]),
    }
    return dets, info, None


def fmt_latlon(lon, lat):
    return f"{abs(lat):.2f}°{'N' if lat >= 0 else 'S'}, {abs(lon):.2f}°{'E' if lon >= 0 else 'W'}"


# ----------------------------------------------------------------------
# Animated map (Leaflet in an iframe)
# ----------------------------------------------------------------------
MAP_TEMPLATE = r"""<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 html,body,#map{margin:0;height:100%;background:#0b2239;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif}
 .card{background:rgba(11,26,43,.88);color:#e6edf5;padding:10px 12px;border-radius:8px;font-size:13px;line-height:1.45;max-width:270px;box-shadow:0 1px 4px rgba(0,0,0,.4)}
 .card b{color:#fff}.card .q{color:#9fb3c8;font-size:12px}
 .legend div{display:flex;align-items:center;gap:6px;margin:0}
 .card.legend{display:flex;flex-wrap:wrap;align-items:center;gap:4px 16px;max-width:calc(100vw - 20px);box-sizing:border-box;padding:6px 10px;font-size:12px;line-height:1.3}
 .card.legend .q{font-size:11px}
 .leaflet-bottom.leaflet-left .leaflet-control{margin-bottom:22px}   /* sits just above the attribution line */
 .sw{width:14px;height:10px;border-radius:2px;display:inline-block}
 .mini{float:right;background:none;border:0;color:#9fb3c8;cursor:pointer;font-size:16px;line-height:1;padding:0 0 0 10px}
 .mini:hover{color:#fff}
 .card.min .more{display:none}
 .btn{background:#ffc400;color:#111;border:0;border-radius:6px;padding:6px 10px;font-weight:600;cursor:pointer;font-size:12px}
 .leaflet-tooltip.iw{background:#ffc400;color:#111;border:0;font-weight:600;font-size:12px;padding:2px 6px;box-shadow:none}
 .leaflet-tooltip.iw.land{background:#9ca3af}
 .leaflet-tooltip.iw:before{display:none}
 .leaflet-tooltip.lm{background:rgba(11,26,43,.88);color:#fff;border:0;font-size:11px;padding:2px 6px;box-shadow:none}
 .leaflet-tooltip.lm:before{display:none}
</style></head><body><div id="map"></div><script>
const P = __PAYLOAD__;
const ll = pts => pts.map(p => [p[1], p[0]]);
const map = L.map('map', {zoomSnap: 0.25, worldCopyJump: false, preferCanvas: true, maxZoom: 13});

// Base maps. maxNativeZoom = the deepest level the provider really has tiles for in open
// water; beyond it Leaflet enlarges the last good tile instead of showing "data not available".
const esri = 'https://server.arcgisonline.com/ArcGIS/rest/services/';
const tl = (u, nat, attr) => L.tileLayer(u, {maxNativeZoom: nat, maxZoom: 13, attribution: attr || ''});
const BASE = {
  'Ocean (Esri)': L.layerGroup([
    tl(esri + 'Ocean/World_Ocean_Base/MapServer/tile/{z}/{y}/{x}', 10, 'Tiles &copy; Esri, GEBCO, NOAA, Garmin, HERE'),
    tl(esri + 'Ocean/World_Ocean_Reference/MapServer/tile/{z}/{y}/{x}', 10)]),
  'Satellite (Esri)': L.layerGroup([
    tl(esri + 'World_Imagery/MapServer/tile/{z}/{y}/{x}', 12, 'Imagery &copy; Esri, Maxar, Earthstar Geographics'),
    tl(esri + 'Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}', 12)]),
  'Light map (CARTO)': L.tileLayer('https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png',
    {subdomains: 'abcd', maxNativeZoom: 14, maxZoom: 13,
     attribution: '&copy; OpenStreetMap contributors &copy; CARTO'})
};
BASE['Ocean (Esri)'].addTo(map);
L.control.layers(BASE, null, {position: 'topleft'}).addTo(map);

const roi = [[P.roi[1], P.roi[0]], [P.roi[3], P.roi[2]]];
L.rectangle(roi, {color: '#ffffff', weight: 1.2, dashArray: '6 6', fill: false, interactive: false}).addTo(map);

// Other detections: only drawn when zoomed out, never on top of this tile's own boxes.
const spots = L.layerGroup();
const tileBox = P.tile ? L.latLngBounds(ll(P.tile.tile_fp)) : null;
P.spots.forEach(s => {
  if (tileBox && tileBox.contains([s[0], s[1]])) return;
  L.circleMarker([s[0], s[1]], {radius: 3.5, color: '#0e7490', weight: 1, fillColor: '#22d3ee', fillOpacity: .85})
   .bindTooltip('Internal wave ' + Math.round(s[2] * 100) + '%').addTo(spots);
});
let spotMaxZoom = null;                       // null = always visible (no tile to focus on)
const viewB = P.tile ? L.latLngBounds(P.tile.view) : null;
const PAD = [40, 40], ZMAX = 10;
// keep the tile and its boxes in the part of the map not covered by the info card (top right)
// and the legend (bottom left); on a small map fall back to even padding
function fitPad() {
  const sz = map.getSize(), c = document.getElementById('card'), g = document.getElementById('legend');
  const ch = c ? c.offsetHeight : 40, lh = g ? g.offsetHeight : 0;
  if (sz.x < 300 || sz.y - ch - lh < 200) return {padding: PAD};
  return {paddingTopLeft: [60, ch + 24], paddingBottomRight: [24, lh + 24]};
}
function padPoint() { const f = fitPad(); return f.padding ? L.point(2 * PAD[0], 2 * PAD[1])
  : L.point(f.paddingTopLeft[0] + f.paddingBottomRight[0], f.paddingTopLeft[1] + f.paddingBottomRight[1]); }
function armSpots() { if (viewB) spotMaxZoom = Math.min(map.getBoundsZoom(viewB, false, padPoint()), ZMAX) - 0.75; }
function updateSpots() {
  const show = spotMaxZoom === null || map.getZoom() <= spotMaxZoom;
  if (show) { if (!map.hasLayer(spots)) spots.addTo(map); }
  else if (map.hasLayer(spots)) map.removeLayer(spots);
}
map.on('zoom zoomend', updateSpots);
let userMoved = false, settled = false;
map.on('resize', () => { if (viewB && settled && !userMoved) {
  map.fitBounds(viewB, Object.assign({maxZoom: ZMAX, animate: false}, fitPad())); armSpots(); updateSpots(); } });
['mousedown', 'wheel', 'touchstart', 'keydown'].forEach(ev =>
  map.getContainer().addEventListener(ev, () => { userMoved = true; }, {passive: true}));

// legend
const legend = L.control({position: 'bottomleft'});
legend.onAdd = () => { const d = L.DomUtil.create('div', 'card legend'); d.id = 'legend'; d.innerHTML =
  '<div><span class="sw" style="border:1.5px dashed #fff"></span>Study region</div>' +
  '<div><span class="sw" style="background:#22d3ee;border-radius:50%;width:10px"></span>' +
    (P.tile ? 'Our other detections (' + P.spots.length + '): zoom out to see' : 'Detections in our scenes (' + P.spots.length + ')') + '</div>' +
  (P.tile ? '<div><span class="sw" style="border:2px solid #fff"></span>This tile (25 km)</div>' +
  '<div><span class="sw" style="background:#ffc400"></span>Detected internal wave</div>' +
  (P.tile.boxes.some(b => b.near_land) ? '<div><span class="sw" style="background:#9ca3af"></span>Near land (likely false)</div>' : '') : '') +
  '<div class="q">' + P.spots_note + '</div>'; return d; };
legend.addTo(map);

// info card
const card = L.control({position: 'topright'});
card.onAdd = () => { const d = L.DomUtil.create('div', 'card'); d.id = 'card'; L.DomEvent.disableClickPropagation(d); return d; };
card.addTo(map);
const cardEl = document.getElementById('card');

const overlay = L.layerGroup().addTo(map);
const reduce = window.matchMedia('(prefers-reduced-motion: reduce)').matches;

// run fn once the map stops moving (with a fallback in case no event fires)
function after(fn, fallbackMs) { let done = false; const go = () => { if (!done) { done = true; fn(); } };
  map.once('moveend', go); setTimeout(go, fallbackMs); }

function drawScene() { if (P.tile.scene_fp) L.polygon(ll(P.tile.scene_fp), {color: '#cbd5e1', weight: 1,
  dashArray: '3 5', fill: false, interactive: false}).addTo(overlay); }
function drawTile() { L.polygon(ll(P.tile.tile_fp), {color: '#ffffff', weight: 2, fill: false, interactive: false}).addTo(overlay); }
function drawLandmarks() {
  (P.tile.landmarks || []).forEach((m, i) => {
    if (i === 0) L.polyline([P.tile.center_ll, [m.lat, m.lon]], {color: '#ffffff', weight: 1.2, dashArray: '2 6',
      opacity: .8, interactive: false}).addTo(overlay);
    L.circleMarker([m.lat, m.lon], {radius: i ? 4 : 5, color: '#0b1a2b', weight: 1.5, fillColor: '#ffffff', fillOpacity: 1})
     .bindTooltip(m.name + ' · ' + m.km + ' km', {permanent: true, direction: i % 2 ? 'left' : 'right',
                  offset: [i % 2 ? -6 : 6, 0], className: 'lm'})
     .addTo(overlay);
  });
}
function drawBoxes(stagger) {
  const many = P.tile.boxes.length > 6;
  P.tile.boxes.forEach((b, i) => setTimeout(() => {
    const c = b.near_land ? '#9ca3af' : '#ffc400';
    const lbl = 'Internal wave ' + Math.round(b.score * 100) + '%' + (b.near_land ? ' · near land' : '');
    L.polygon(ll(b.poly), {color: c, weight: 2.5, fillColor: c, fillOpacity: .18, dashArray: b.near_land ? '5 4' : null})
     .bindTooltip(lbl, {permanent: !many, direction: 'top', className: 'iw' + (b.near_land ? ' land' : '')})
     .addTo(overlay);
  }, stagger ? i * 260 : 0));
}
function showCard() {
  const t = P.tile, n = t.boxes.length;
  settled = true;
  cardEl.classList.add('min');          // collapsed by default; "+" opens it for a demo
  cardEl.innerHTML = '<button class="mini" id="mini" title="Expand">+</button>' +
    '<b>' + (t.time || 'Sentinel-1 scene') + '</b><div class="more">' +
    (t.place_text ? '<span id="pl"></span><br>' : '') +
    'Tile centre ' + t.center_txt + '<br>' +
    (n ? n + ' internal-wave box' + (n > 1 ? 'es' : '') + ' at &ge; ' + P.threshold_pct + '%'
       : 'No boxes at &ge; ' + P.threshold_pct + '%') +
    '<div class="q" style="margin-top:6px">Position from the scene\'s ground-control points; approximate.</div>' +
    '<div style="margin-top:8px"><button class="btn" id="replay">&#8635; Replay</button></div></div>';
  if (t.place_text) document.getElementById('pl').textContent = t.place_text;    // text, never HTML
  document.getElementById('replay').onclick = play;
  const mini = document.getElementById('mini');
  mini.onclick = () => { const m = cardEl.classList.toggle('min');
    mini.innerHTML = m ? '+' : '&minus;'; mini.title = m ? 'Expand' : 'Minimise'; };
}

function play() {
  settled = false; userMoved = false;
  overlay.clearLayers(); cardEl.classList.remove('min'); cardEl.innerHTML = '<span class="q">Locating tile&hellip;</span>';
  map.fitBounds(roi, {animate: false, padding: [10, 10]});
  armSpots(); updateSpots();
  setTimeout(() => {
    const sceneB = P.tile.scene_fp ? L.latLngBounds(ll(P.tile.scene_fp)) : null;
    const toTile = () => { map.flyToBounds(viewB, Object.assign({duration: 1.8, maxZoom: ZMAX}, fitPad()));
      after(() => { drawTile(); drawLandmarks(); drawBoxes(true); showCard(); }, 2600); };
    if (sceneB) { map.flyToBounds(sceneB, {duration: 1.6, padding: [30, 30]});
      after(() => { drawScene(); setTimeout(toTile, 350); }, 2300); }
    else toTile();
  }, 650);
}

if (!P.tile) {
  map.fitBounds(roi, {padding: [10, 10]});
  updateSpots();
  cardEl.innerHTML = '<b>Location unknown</b><br><span class="q"></span>';
  cardEl.querySelector('.q').textContent = P.message.replace(/`/g, '');   // text, never HTML
} else if (P.animate && !reduce) {
  play();
} else {
  map.fitBounds(viewB, Object.assign({maxZoom: ZMAX, animate: false}, fitPad()));
  armSpots(); updateSpots();
  drawScene(); drawTile(); drawLandmarks(); drawBoxes(false); showCard();
}
</script></body></html>"""


def render_geomap(dets, tile_info, message, threshold, animate, height=600):
    spots, note = load_detections(files_stamp(DETECTIONS_V2_PATH, DETECTIONS_OLD_PATH))
    payload = {
        "roi": ROI,
        "spots": spots[["real_lat", "real_lon", "confidence"]].round(5).values.tolist(),
        "spots_note": f"Detection spots: {note}",
        "threshold_pct": int(round(threshold * 100)),
        "animate": bool(animate),
        "message": message or "",
        "tile": None,
    }
    if tile_info is not None:
        payload["tile"] = {
            "scene_fp": tile_info["scene_fp"], "tile_fp": tile_info["tile_fp"],
            "time": tile_info["time"],
            "center_txt": fmt_latlon(*tile_info["center"]),
            "center_ll": [tile_info["center"][1], tile_info["center"][0]],
            "place_text": tile_info["place"]["text"] if tile_info.get("place") else "",
            "view": tile_info["view"],
            "landmarks": [{"name": m["name"], "km": int(round(m["km"])), "lat": m["lat"], "lon": m["lon"]}
                          for m in tile_info.get("landmarks", [])],
            "boxes": [{"poly": d["poly"], "score": d["score"], "near_land": d["near_land"]}
                      for d in dets],
        }
    html = MAP_TEMPLATE.replace("__PAYLOAD__", json.dumps(payload).replace("</", "<\\/"))
    if hasattr(st, "iframe"):          # newer Streamlit (components.html is being retired)
        st.iframe(html, height=height)
    else:
        components.html(html, height=height)


# ----------------------------------------------------------------------
# Static SST / wind map (tab 2), drawn once
# ----------------------------------------------------------------------
LANDMARKS = [
    ("Chennai", 80.27, 13.08, False), ("Kolkata", 88.36, 22.57, False),
    ("Visakhapatnam", 83.30, 17.69, False), ("Colombo", 79.86, 6.93, False),
    ("Port Blair", 92.73, 11.62, False), ("Andaman-Nicobar ridge", 93.0, 10.0, True),
    ("Bay of Bengal", 87.0, 15.0, True), ("Andaman Sea", 95.5, 12.0, True),
]


@st.cache_data(show_spinner="Drawing the map (first time only)...")
def multimodal_map_png(stamp=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    df = pd.read_parquet(SPATIAL_MATCHES_PATH).dropna(
        subset=["real_lon", "real_lat", "sst_sst", "era5_u10", "era5_v10"])
    det, note = load_detections(files_stamp(DETECTIONS_V2_PATH, DETECTIONS_OLD_PATH))
    sst_c = df["sst_sst"] - 273.15
    sub = df.iloc[::max(1, len(df) // 350)]

    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        fig = plt.figure(figsize=(11, 9))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([ROI[0], ROI[2], ROI[1], ROI[3]], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="#dbeafe")
        ax.add_feature(cfeature.LAND, facecolor="#e8e0d0")
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="#555")
        kw = {"transform": ccrs.PlateCarree()}
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)
        gl.top_labels = gl.right_labels = False
    except ImportError:
        fig, ax = plt.subplots(figsize=(11, 9))
        ax.set_facecolor("#dbeafe")
        ax.set_xlim(ROI[0], ROI[2]); ax.set_ylim(ROI[1], ROI[3]); ax.set_aspect("equal")
        kw = {}

    sc = ax.scatter(df["real_lon"], df["real_lat"], c=sst_c, cmap="RdYlBu_r", s=8, alpha=0.6,
                    edgecolors="none", vmin=26, vmax=31, zorder=3, **kw)
    ax.quiver(sub["real_lon"].values, sub["real_lat"].values, sub["era5_u10"].values,
              sub["era5_v10"].values, color="black", alpha=0.35, scale=160, width=0.0018,
              zorder=4, **kw)
    ax.scatter(det["real_lon"], det["real_lat"], s=20, c="lime", edgecolors="darkgreen",
               linewidths=0.4, alpha=0.85, label=f"Internal wave detections ({len(det)})",
               zorder=5, **kw)
    for name, lon, lat, water in LANDMARKS:
        color = "navy" if water else "black"
        ax.plot(lon, lat, marker="*" if water else "^", color=color, markersize=11, zorder=6, **kw)
        ax.text(lon + 0.2, lat + 0.2, name, fontsize=9, color=color, zorder=6,
                style="italic" if water else "normal", **kw)
    fig.colorbar(sc, ax=ax, label="Sea surface temperature (°C)", shrink=0.7)
    ax.legend(loc="upper right", framealpha=0.9)
    ax.set_title("June 2021: SST (colour) + ERA5 wind (arrows) + internal-wave detections (green)")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue(), note


# ----------------------------------------------------------------------
# Conditions at scene time (live weather, with an offline fallback)
# ----------------------------------------------------------------------
@st.cache_resource
def _conditions_store():
    return {}


def get_conditions(lat, lon, time_text):
    """-> (values or None, error text or None). Successes are kept for the life of the
    server; failures for 2 minutes, so being offline doesn't slow every click."""
    import time as _time
    when = ctx.parse_scene_time(time_text)
    if when is None:
        return None, "scene time unknown"
    key = (round(lat * 4) / 4, round(lon * 4) / 4, ctx._nearest_hour(when).isoformat())
    store = _conditions_store()
    hit = store.get(key)
    if hit and (hit[1] is not None or _time.time() - hit[0] < 120):
        return hit[1], hit[2]
    cond, err = ctx.fetch_conditions(lat, lon, when)
    store[key] = (_time.time(), cond, err)
    return cond, err


@st.cache_data
def load_matches(stamp=None):
    cols = ["real_lon", "real_lat", "era5_u10", "era5_v10", "sst_sst"]
    return pd.read_parquet(SPATIAL_MATCHES_PATH, columns=cols).dropna().to_numpy(dtype=float)


def monthly_nearby(lat, lon, radius_deg=0.5):
    """Offline fallback: June 2021 monthly ERA5 wind and SST from our own matched patches
    within ~55 km. -> values or None."""
    if not os.path.exists(SPATIAL_MATCHES_PATH):
        return None
    try:
        a = load_matches(files_stamp(SPATIAL_MATCHES_PATH))
    except Exception:
        return None
    near = a[(a[:, 0] - lon) ** 2 + (a[:, 1] - lat) ** 2 <= radius_deg ** 2]
    if len(near) == 0:
        return None
    u, v = float(near[:, 2].mean()), float(near[:, 3].mean())
    return {"wind_ms": float(np.hypot(u, v)),
            "wind_dir": float((270 - np.degrees(np.arctan2(v, u))) % 360),
            "sst_c": float(near[:, 4].mean() - 273.15), "n": int(len(near))}


def short_reason(err):
    e = (err or "").lower()
    if any(k in e for k in ("proxy", "connection", "timeout", "resolve", "network", "unreachable")):
        return "no connection to the weather service"
    return "the weather service returned an error"


def show_seafloor(tile_info):
    """What the sea floor looks like under this tile (GEBCO), in plain language."""
    if tile_info is None or bathy is None:
        return
    lon, lat = tile_info["center"]
    sf = bathy.describe_seafloor(lat, lon)
    if sf is None or sf["depth_m"] <= 0:
        return
    st.subheader("Sea floor under this tile")
    c1, c2, c3 = st.columns(3)
    c1.metric("Water depth", f"{sf['depth_m']:,.0f} m")
    c2.metric("Zone", sf["zone"].capitalize())
    sb = sf["shelf_break_km"]
    c3.metric("Distance to shelf edge (200 m)", "n/a" if sb is None else f"{sb:.0f} km")
    notes = []
    if sf["slope_m_per_km"] >= 40:
        notes.append("The sea floor is steep nearby. Tides pushed over steep slopes, ridges and shelf "
                     "edges are a common place for internal waves to form, so this fits what the detector found.")
    elif sf["zone"] in ("very shallow shelf", "continental shelf"):
        notes.append("Shallow, gently sloping shelf. Internal waves here usually arrived from deeper water "
                     "or formed at the shelf edge and travelled in.")
    else:
        notes.append("Open deep water. Internal waves here usually travel a long way from where they formed.")
    for n in notes:
        st.markdown(f"- {n}")
    st.caption("Depth from the GEBCO global grid, resampled to about 5.5 km, so narrow canyons and "
               "small features are smoothed out. GEBCO Compilation Group.")


def show_conditions(tile_info):
    """Wind, waves and sea temperature for this tile's date, plus a plain-language reading."""
    if tile_info is None or ctx is None or not tile_info.get("time"):
        return
    st.subheader("Conditions when this was captured")
    lon, lat = tile_info["center"]
    with st.spinner("Looking up the weather for this date and place..."):
        cond, err = get_conditions(lat, lon, tile_info["time"])
    monthly = None
    if cond is None:
        monthly = monthly_nearby(lat, lon)
        shown = monthly
    else:
        shown = cond
    if shown is None:
        st.info(f"Weather for this date isn't available right now: {short_reason(err)}.")
        if err:
            with st.expander("Technical detail"):
                st.code(err)
        return

    def num(v, fmt, unit=""):
        return "n/a" if v is None else f"{v:{fmt}}{unit}"
    wind_dir = shown.get("wind_dir")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Wind" + (f" (from {ctx.compass(wind_dir)})" if wind_dir is not None else ""),
              num(shown.get("wind_ms"), ".1f", " m/s"))
    if monthly is None:
        c2.metric("Waves (height · period)",
                  "n/a" if shown.get("wave_h") is None else
                  f"{shown['wave_h']:.1f} m · {num(shown.get('wave_t'), '.0f', ' s')}")
        c4.metric("Cloud · rain", f"{num(shown.get('cloud_pct'), '.0f', '%')} · {num(shown.get('rain_mm'), '.1f', ' mm/h')}")
    else:
        c2.metric("Waves (height · period)", "n/a")
        c4.metric("Cloud · rain", "n/a")
    c3.metric("Sea-surface temperature", num(shown.get("sst_c"), ".1f", " °C"))

    if monthly is None:
        st.caption(f"{cond['source']} · nearest hour {cond['hour_utc'].replace('T', ' ')} UTC · about 25 km grid cell at "
                   "the tile centre. Reanalysis values (a model constrained by observations), not a "
                   "measurement at the exact spot. Weather data by Open-Meteo.com (CC BY 4.0).")
        if err:                      # e.g. waves or sea temperature were unavailable
            with st.expander("Some values were unavailable"):
                st.code(err)
    else:
        st.caption(f"Live weather unavailable ({short_reason(err)}). Showing the June 2021 monthly average "
                   f"from our own ERA5 / SST data within ~55 km ({monthly['n']} patches), not this exact day.")
        if err:
            with st.expander("Technical detail"):
                st.code(err)
    for line in ctx.interpret(shown if monthly is None else {}, tile_info.get("place")):
        st.markdown(f"- {line}")


# ----------------------------------------------------------------------
# Page
# ----------------------------------------------------------------------
st.markdown("""<style>
/* tighter page: less empty space above the title and between blocks */
.block-container {padding-top: 1.2rem; padding-bottom: 1.5rem;}
div[data-testid="stVerticalBlock"] {gap: 0.6rem;}
/* centred heading and one-line descriptions */
.ox-title {text-align: center; margin: 0 0 0.2rem 0; font-size: 1.85rem; font-weight: 700; line-height: 1.2;}
.ox-desc {text-align: center; color: rgba(128,128,128,0.95); font-size: 0.9rem; margin: 0 0 0.4rem 0;}
/* detection image shrinks with the screen height, so the table and map stay in view */
.st-key-ox_left div[data-testid="stImage"] img {max-height: max(220px, calc(100vh - 560px)); width: auto !important;
                                                 max-width: 100%; object-fit: contain; margin: 0 auto; display: block;}
.ox-hint {font-size: 0.78rem; line-height: 1.3; color: rgba(128,128,128,0.95); margin: -0.5rem 0 0.4rem 0;}
/* live map fills the rest of the screen height (Leaflet re-fits on resize) */
.st-key-ox_right iframe {height: max(380px, calc(100vh - 290px)) !important;}
.st-key-ox_right [data-testid="stElementContainer"]:has(> iframe) {height: auto !important;}
.st-key-ox_right [data-testid="stCaptionContainer"] {margin-top: -0.5rem;}
</style>""", unsafe_allow_html=True)


def centered_title(text):
    st.markdown(f'<h1 class="ox-title">{html_escape(text)}</h1>', unsafe_allow_html=True)


def centered_desc(text):
    st.markdown(f'<p class="ox-desc">{html_escape(text)}</p>', unsafe_allow_html=True)


centered_title("OceanXRay -- Multimodal Ocean Dashboard")

if not os.path.exists(CHECKPOINT_PATH):
    st.error(f"Checkpoint not found at '{CHECKPOINT_PATH}'. Place ckpt_improved_v2_s2.pth "
             "in the same folder as this script.")
    st.stop()
load_model()

UPLOAD_HELP = ("Upload a Sentinel-1 tile (from convert_scenes_to_images.py, keep the original "
               "file name to geolocate it) or an S1-IW-2023 test image")
MAP_HEIGHT = 600
threshold = 0.70          # fixed confidence threshold (no slider)

tab1, tab2 = st.tabs(["Internal wave detector (live)", "Integrated multimodal map"])

with tab1:
    centered_desc("Faster R-CNN v2 (improved training), fine-tuned on S1-IW-2023: "
                  "92.3% mAP@0.5 on the test split (mean of 3 runs).")
    # Stacked layout: upload, then detection, on the left; the live map uses the right half.
    left, right = st.columns([5, 6], gap="medium")
    try:                                   # keyed containers let the CSS above target each side
        ox_left = left.container(key="ox_left")
        ox_right = right.container(key="ox_right")
    except TypeError:                      # older Streamlit: same layout, default sizes
        ox_left, ox_right = left.container(), right.container()
    upload_key = f"tile_upload_{st.session_state.get('upload_round', 0)}"
    with ox_left:
        try:
            upload_box = st.container(key="ox_upload")
        except TypeError:
            upload_box = st.container()
        with upload_box:
            uploaded = st.file_uploader(UPLOAD_HELP, type=["png", "jpg", "jpeg"],
                                        label_visibility="collapsed", key=upload_key)
            st.markdown(f'<p class="ox-hint">{html_escape(UPLOAD_HELP)}</p>', unsafe_allow_html=True)
        if uploaded is not None:
            # detection showing: hide the upload box and its instructions to give the image the space
            st.markdown("<style>.st-key-ox_upload {display: none;}</style>", unsafe_allow_html=True)

            def _new_tile():
                st.session_state["upload_round"] = st.session_state.get("upload_round", 0) + 1
            st.button("Upload another tile", on_click=_new_tile)

    if uploaded is None:
        with ox_left:
            st.info("Upload a tile to run detection.")
    else:
        data = uploaded.getvalue()
        boxes, scores, size = detect(data)
        image = Image.open(io.BytesIO(data)).convert("RGB")
        dets, tile_info, message = geolocate(uploaded.name, size, boxes, scores, threshold)

        with ox_left:
            shown, shown_dets = trim_nodata(image, dets)
            st.image(draw_detections(shown, shown_dets),
                     caption=f"{len(dets)} detection(s) at ≥ {threshold:.0%}",
                     use_container_width=True)
            with st.expander("Input"):
                st.image(image, caption="Input", use_container_width=True)
            if dets:
                def place_cell(d):
                    if d.get("place"):
                        text = d["place"]["text"]
                    elif d.get("center"):
                        text = fmt_latlon(*d["center"])
                    else:
                        text = "unknown"
                    return text + ("  [within 1 km of land, likely false]" if d["near_land"] else "")
                table = pd.DataFrame([{
                    "Confidence": f"{d['score']:.0%}",
                    "Place": place_cell(d),
                    "Coordinates": fmt_latlon(*d["center"]) if d.get("center") else "unknown",
                } for d in sorted(dets, key=lambda d: -d["score"])])
                table.index = range(1, len(table) + 1)
                # fixed, short height: many detections scroll inside the table, not the page
                st.dataframe(table, height=min(38 + 35 * len(table), 150),
                             use_container_width=True)
            else:
                st.info("No detections above the current threshold.")

        with ox_right:
            st.markdown("**Where is this tile?**")
            if tile_info is not None and tile_info.get("place"):
                st.caption(f"Tile centre: **{tile_info['place']['text']}** "
                           f"({fmt_latlon(*tile_info['center'])}).  Zoom out on the map to see our other detections.")
            # Fly-in animation once per uploaded file; later reruns (e.g. moving
            # the slider) redraw at the tile view without replaying it.
            key = hashlib.md5(data).hexdigest() + uploaded.name
            animate = st.session_state.get("animated_key") != key
            st.session_state["animated_key"] = key
            render_geomap(dets, tile_info, message, threshold, animate, height=MAP_HEIGHT)

        # Below the first screen, full width: sea floor and weather for this tile
        show_seafloor(tile_info)
        show_conditions(tile_info)

with tab2:
    centered_desc("SST, ERA5 wind and internal-wave detections together, June 2021. ERA5 and SST "
                  "are monthly data, so this is a monthly-representative view, not one day.")
    if not os.path.exists(SPATIAL_MATCHES_PATH):
        st.warning(f"Map data not found: place {SPATIAL_MATCHES_PATH} next to this script.")
    else:
        png, note = multimodal_map_png(files_stamp(
            SPATIAL_MATCHES_PATH, DETECTIONS_V2_PATH, DETECTIONS_OLD_PATH))
        st.image(png, use_container_width=True)
        st.caption(f"Detection spots: {note}.")

Writing /kaggle/working/iw_dashboard.py


In [8]:
import os
import subprocess

cloudflared_path = "/kaggle/working/cloudflared"

if not os.path.exists(cloudflared_path):
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", cloudflared_path
        ],
        check=True
    )

os.chmod(cloudflared_path, 0o755)

print("cloudflared is ready.")

cloudflared is ready.


In [9]:
import os, sys, subprocess, time, re, requests

PORT = 8501
APP_PATH = "/kaggle/working/iw_dashboard.py"
GEO_DIR = "/kaggle/input/datasets/mayankgupta005/ocean-geo"   # holds oceanxray_geo.py

# stop the broken run and any old tunnel (brackets stop pkill matching itself)
subprocess.run("pkill -f 'streamlit ru[n]'; pkill -f '[c]loudflared tunnel'", shell=True)
time.sleep(2)

env = dict(os.environ, PYTHONPATH="/kaggle/working:" + GEO_DIR + ":" + os.environ.get("PYTHONPATH", ""))
st_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP_PATH,
     f"--server.port={PORT}", "--server.address=127.0.0.1", "--server.headless=true",
     "--server.enableCORS=false", "--server.enableXsrfProtection=false",
     "--server.fileWatcherType=none", "--browser.gatherUsageStats=false"],
    stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT,
    env=env, start_new_session=True)

for _ in range(40):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2).text == "ok":
            break
    except requests.RequestException:
        time.sleep(1)

tun = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url",
                        f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                       stdout=open("/tmp/cloudflared.log", "w"),
                       stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(40):
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if m:
        print("Open:", m.group(0)); break
    time.sleep(1)

Open: https://characteristic-hughes-ebooks-lips.trycloudflare.com
